# Chit 6 baseline: a heartbeat classifier for a wearable · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a small 1-D CNN on single beats, measure macro-F1 on **unseen patients** in fp32 and with int8 weights and activations, and count what it costs on a microcontroller: flash, SRAM, MACs, and energy per beat.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime", "wfdb"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data
One beat = 256 samples (0.71 s at 360 Hz) of lead MLII around the R peak. Classes N, S, V, F, Q (AAMI). No patient is in two splits.

In [ ]:
from capstone.data.mitbih import load, CLASSES, macro_f1
X_tr, y_tr = load("train"); X_va, y_va = load("val"); X_te, y_te = load("test")
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te)):
    print(f"{name:<5} {len(y):>6,} beats  " + "  ".join(f"{c} {np.mean(y == k):6.2%}" for k, c in enumerate(CLASSES)))

We subtract each beat's mean (removes the slow baseline drift; 256 additions on the device, negligible).

In [ ]:
prep = lambda X: (X - X.mean(-1, keepdims=True)).astype(np.float32)
XTR, XVA, XTE = prep(X_tr), prep(X_va), prep(X_te)
if FAST:
    keep = np.random.default_rng(0).permutation(len(XTR))[:10_000]; XTR, y_tr = XTR[keep], y_tr[keep]

## 2. The model, costed before training
Three strided 1-D convolutions, global average pooling, one linear layer.

In [ ]:
model = nn.Sequential(nn.Conv1d(1, 8, 7, stride=2, padding=3), nn.ReLU(),
                      nn.Conv1d(8, 16, 5, stride=2, padding=2), nn.ReLU(),
                      nn.Conv1d(16, 32, 5, stride=2, padding=2), nn.ReLU(),
                      nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(32, 5))
p = profile(model, (1, 256))
TARGET = dict(max_weight_bytes=64 * 1024, max_act_bytes=16 * 1024, mac_rate=100e6)   # DRAFT values from chits.md; add max_latency_s once ⟨T⟩ is set
print(f"params {p['params']:,} | MACs {p['macs']:,} | int8 weights {p['weight_bytes']:,} B | peak activations {p['peak_act_bytes']:,} B")
print(f"latency at 100 M MAC/s: {1e3 * mcu_latency(p['macs'], 100e6):.3f} ms per beat | energy per beat (card formula): {energy_uj(p):.2f} µJ")
fits(p, TARGET);

## 3. Train (class-weighted: N is ~90% of the beats) and measure

In [ ]:
w = 1 / np.sqrt(np.bincount(y_tr, minlength=5) + 1); w = w / w.mean()
fit(model, XTR, y_tr, epochs=2 if FAST else 15, bs=256, lr=3e-3, class_weight=w)
F1_32 = macro_f1(y_te, predict(model, XTE).argmax(1))
print(f"macro-F1 on test patients, fp32: {F1_32:.4f}   (val: {macro_f1(y_va, predict(model, XVA).argmax(1)):.4f})")

## 4. int8 (weights and activations), with ONNX Runtime static quantization
An MCU runs int8: this is the number the chit counts.

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/mitbih_fp32.onnx")
ort_int8("my_models/mitbih_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/mitbih_int8.onnx")
sess8 = ort_session("my_models/mitbih_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

F1_8 = macro_f1(y_te, OUT8.argmax(1))
from sklearn.metrics import f1_score
print(f"macro-F1 on test patients, int8: {F1_8:.4f}")
print("per-class F1 (int8):", dict(zip(CLASSES, np.round(f1_score(y_te, OUT8.argmax(1), labels=range(5), average=None, zero_division=0), 3))))

In [ ]:
E = energy_uj(p)
ROW = report("6 mitbih", (len(XTR), len(XVA), len(XTE)), (1, 256), f"macro-F1 int8 {F1_8:.4f} (fp32 {F1_32:.4f})", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.3f} ms at 100 M MAC/s; energy {E:.2f} µJ/beat",
             "ODC-By 1.0 (PhysioNet, confirmed)", extra=dict(energy_uj=round(E, 3), minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.